# Machine Learning and Natural Language Processing
## Esercizio 2. Dal testo alla tassonomia: un decoder estrae, un encoder collega

**Master AIDA, Edizione XVI, 2026.** Giuliano Lorenzo, PhD Student in Economics, Statistics and Data Science

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/giulianolorenzo-PhD/aida-ml-nlp/blob/main/esercizio_2.ipynb)

**Il problema.** Un'agenzia per il lavoro riceve migliaia di annunci scritti liberamente: "esperienza con Postgres",
"conoscenza di PostgreSQL", "DBA PostgreSQL" indicano la stessa competenza. Per confrontare annunci e candidati
o contare le competenze richieste dal mercato serve un **vocabolario comune**: una **tassonomia**.

Usiamo **ESCO** (European Skills, Competences, Qualifications and Occupations), la tassonomia ufficiale
dell'Unione Europea, e dividiamo il lavoro tra due tipi di modelli:

| Modello | Come legge | Compito |
|:--|:--|:--|
| **Decoder** (un LLM generativo) | genera testo un token alla volta, seguendo istruzioni | **estrarre** le competenze dalla frase e **scegliere** tra poche opzioni |
| **Encoder** (un modello di embedding) | legge tutta la frase e la riassume in un vettore | **collegare** ogni competenza alla voce ESCO più vicina nel significato |

Il risultato sono **feature esplicite e standardizzate**: voci ESCO che si possono contare, aggregare e confrontare.

Procediamo per pezzi: prima l'**encoder** da solo (sezione 3), poi il **decoder** (sezione 4), infine le combinazioni (sezione 5).
Per ciascuno partiamo da un modello piccolo e vediamo quanto rende sceglierne uno più forte.

## Indice
0. Setup
1. La tassonomia ESCO
2. I dati: frasi di annunci annotate a mano
3. L'encoder: collegare un testo alla tassonomia
4. Il decoder: estrarre le competenze
5. Quattro strategie a confronto
6. Dalle competenze alle feature
7. Tutta ESCO
8. Bonus: un annuncio in italiano
9. Costi e salvataggi
10. Cosa portare a casa

## 0. Setup

- **Attivate la GPU**: menu *Runtime*, *Cambia tipo di runtime*, *GPU T4*. L'encoder che sceglieremo codifica
  oltre 100.000 nomi di competenze: con la GPU servono circa un paio di minuti, sulla CPU più di un'ora.
- Le risposte dei modelli sono **salvate in un file** (cache) che il notebook scarica dal repository del corso
  (`USE_COURSE_CACHE = True`): gira **senza chiave e senza costi**, ma tempi e costi misurati risultano zero.
  Se una richiesta non è nella cache, senza chiave viene saltata e il notebook lo segnala nella sezione 9.
- Per nuove richieste (altri modelli, altre istruzioni) serve una **chiave OpenRouter** con qualche euro di credito.
  Con `USE_COURSE_CACHE = False` tutte le richieste partono da zero: l'intero notebook costa meno di 20 centesimi di dollaro. In Colab salvatela nei **Secrets** (icona della
  chiave nella barra a sinistra) con il nome `OPENROUTER_API_KEY` e attivate l'accesso per questo notebook.
  **Mai scrivere una chiave nel codice**: chi legge il notebook potrebbe usarla.

In [1]:
%pip install -q openai gensim sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import re
import threading
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from urllib.request import urlopen, urlretrieve

import gensim.downloader
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import torch
from huggingface_hub.utils import logging as hub_logging
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers.utils import logging as transformers_logging

transformers_logging.disable_progress_bar()      # niente barre di avanzamento durante il caricamento dei modelli
hub_logging.set_verbosity_error()                # nasconde gli avvisi tecnici dell'Hugging Face Hub
pio.templates.default = "plotly_white"
pd.set_option("display.max_colwidth", 110)
RANDOM_STATE = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {DEVICE}")

Dispositivo: cuda


Con `USE_DRIVE = True` i salvataggi (cache delle risposte, indici) restano su Google Drive anche dopo la chiusura di Colab.

In [3]:
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = Path("/content/drive/MyDrive/aida_esercizio_2")
else:
    SAVE_DIR = Path("salvataggi")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

COURSE = "https://raw.githubusercontent.com/giulianolorenzo-PhD/aida-ml-nlp/main"     # il repository del corso: da qui la cache
USE_COURSE_CACHE = True        # True: le risposte già pronte del corso; False: tutte le richieste da zero (serve la chiave)
try:                                                   # Colab: la chiave è nei Secrets
    from google.colab import userdata
    API_KEY = userdata.get("OPENROUTER_API_KEY")
except Exception:                                      # altrove, o senza Secret: variabile d'ambiente, se c'è
    API_KEY = os.environ.get("OPENROUTER_API_KEY")
print("Chiave caricata:", bool(API_KEY))
if not API_KEY:
    print("Nessuna chiave: si usano solo le risposte salvate nella cache.")

Chiave caricata: True


## 1. La tassonomia ESCO

Scarichiamo ESCO v1.1.1 in inglese dal **sito ufficiale della Commissione Europea** (un file zip di tabelle CSV). Ne usiamo tre:
- `skills_en.csv`: una riga per competenza, con **etichetta preferita** (preferred label), **sinonimi** (alternative labels) e descrizione;
- `broaderRelationsSkillPillar_en.csv`: la **gerarchia**, cioè quale gruppo contiene quale competenza;
- `skillGroups_en.csv`: i nomi dei gruppi.

In [4]:
ESCO_URL = "https://ec.europa.eu/esco/download/ESCO%20dataset%20-%20v1.1.1%20-%20classification%20-%20en%20-%20csv.zip"
esco_zip_path = SAVE_DIR / "esco_v1.1.1_en.zip"
if not esco_zip_path.exists():
    esco_zip_path.write_bytes(urlopen(ESCO_URL).read())

with zipfile.ZipFile(esco_zip_path) as z:
    esco = pd.read_csv(z.open("skills_en.csv"))
    hierarchy = pd.read_csv(z.open("broaderRelationsSkillPillar_en.csv"))
    groups = pd.read_csv(z.open("skillGroups_en.csv"))

print(f"Competenze in ESCO: {len(esco)}")
esco["skillType"].value_counts()

Competenze in ESCO: 13896


skillType
skill/competence    10831
knowledge            3059
Name: count, dtype: int64

Una competenza da vicino: etichetta preferita, sinonimi e posizione nella gerarchia.

In [5]:
names = pd.concat([esco.set_index("conceptUri")["preferredLabel"], groups.set_index("conceptUri")["preferredLabel"]])
parents = hierarchy.groupby("conceptUri")["broaderUri"].first()     # un genitore per voce (il primo, se sono più di uno)


def path_to_root(uri):
    """Il percorso dalla voce alla radice della gerarchia."""
    path = [uri]
    while path[-1] in parents.index:
        path.append(parents[path[-1]])
    return [names.get(u, u) for u in path]


skill = esco.query("preferredLabel == 'manage a team'").iloc[0]
print("Etichetta preferita:", skill["preferredLabel"])
print("Sinonimi:", skill["altLabels"].replace("\n", " | "))
print("Descrizione:", skill["description"])
print("Gerarchia:", "  <  ".join(path_to_root(skill["conceptUri"])))

Etichetta preferita: manage a team
Sinonimi: managing a team | supervise a team | manage a team | manage the team | direct a team
Descrizione: Ensure clear and effective channels of communication across all departments within the organisation and support functions, both internally and externally ensuring that the team is aware of the standards and objectives of the department/business unit. Implement the disciplinary and grievance procedures as required ensuring that a fair and consistent approach to managing performance is consistently achieved. Assist in the recruitment process and manage, train and motivate employees to achieve/exceed their potential using effective performance management techniques. Encourage and develop a team ethic amongst all employees.
Gerarchia: manage a team  <  supervising a team or group  <  supervising people  <  management skills  <  skills


## 2. I dati: frasi di annunci annotate a mano

Due dataset pubblici di TechWolf (licenza CC BY 4.0) con frasi di annunci di lavoro reali in inglese:
**tech** (informatica) e **house** (settori vari). Ogni riga è una **menzione** di competenza in una frase,
con l'etichetta ESCO assegnata da un annotatore. Due etichette speciali:
- `LABEL NOT PRESENT`: la competenza c'è, ma in ESCO **non esiste** una voce adatta;
- `UNDERSPECIFIED`: la menzione è troppo vaga per assegnarle una voce.

Il **gold standard** di una frase è l'insieme delle sue etichette ESCO (escluse le due speciali).

In [6]:
TECHWOLF = "https://huggingface.co/datasets/TechWolf/skill-extraction-{source}/resolve/{rev}/{split}.csv"
REVISIONS = {"tech": "5f0cb05df03cd1cd5048ff1d64639bf9bd4cb36f", "house": "dcb313cad615a8f5148ef5addc01ef090fc9f295"}
NO_LABEL = {"LABEL NOT PRESENT", "UNDERSPECIFIED"}


def load_mentions(split):
    return pd.concat([pd.read_csv(TECHWOLF.format(source=s, rev=rev, split=split)).assign(source=s)
                      for s, rev in REVISIONS.items()], ignore_index=True)


mentions = {split: load_mentions(split) for split in ["validation", "test"]}
mentions["test"].sample(6, random_state=1)[["source", "sentence", "span", "label"]]

,source,sentence,span,label
1392,house,Have good interpersonal and communication skills both in Danish and English; spoken as written,English;,English
1825,house,Strong leadership skills - YOU are in charge of the group you are taking out Support to our corporate even...,assiting with teambuilding excercises,encourage teambuilding
1574,house,Support Product Owner to identify and manage risks debt issues and opportunities for technical improvement,identify and manage risks debt issues and opportunities,LABEL NOT PRESENT
655,tech,Team player with a methodical and reliable approach to problem solving,methodical and reliable approach,LABEL NOT PRESENT
845,tech,• Understand operational processes in order to provide solutions which improve operational efficiency,operational processes,operations department processes
1767,house,We are particularly interested in research contributing to business research .,business research .,perform business research


La **validazione** (178 frasi) serve a prendere le decisioni (modelli, soglie). Per contenere tempi e costi,
il confronto finale usa **150 frasi di test** estratte a caso (75 per dataset).

In [7]:
def to_sentences(m):
    """Una riga per frase, con l'insieme delle etichette ESCO corrette."""
    return (m.groupby(["source", "sentence"], sort=False)["label"]
            .agg(lambda labels: set(labels) - NO_LABEL).rename("gold").reset_index())


val = to_sentences(mentions["validation"])
test = to_sentences(mentions["test"]).groupby("source").sample(75, random_state=RANDOM_STATE).reset_index(drop=True)
SETS = {"val": val, "test": test}

print(f"Frasi di validazione: {len(val)}, di test: {len(test)}")
print(f"Etichette ESCO per frase (test): media {test['gold'].str.len().mean():.1f}, "
      f"frasi senza etichette ESCO: {(test['gold'].str.len() == 0).sum()}")

Frasi di validazione: 178, di test: 150
Etichette ESCO per frase (test): media 1.4, frasi senza etichette ESCO: 34


#### Un campione della tassonomia, scelto con un criterio

Per l'esercitazione usiamo un sottoinsieme di ESCO, **non casuale**:
1. tutte le competenze **corrette** delle nostre frasi;
2. tutte le loro **sorelle**: le competenze nello **stesso gruppo** della gerarchia.

Le sorelle sono le alternative più insidiose: "PostgreSQL" compete con "MySQL", "Oracle Relational Database" e gli altri database.
Un campione casuale conterrebbe soprattutto competenze lontane e facili da scartare.
Il campione è comunque costruito guardando le etichette corrette, quindi è un po' ottimista:
nella sezione 7 ripetiamo la strategia migliore su **tutta ESCO**.

In [8]:
label_to_uri = esco.set_index("preferredLabel")["conceptUri"]
gold_uris = set(label_to_uri[list(set().union(*val["gold"], *test["gold"]))])
gold_parents = hierarchy.loc[hierarchy["conceptUri"].isin(gold_uris), "broaderUri"]
sibling_uris = set(hierarchy.loc[hierarchy["broaderUri"].isin(gold_parents), "conceptUri"])

taxonomy = esco[esco["conceptUri"].isin(gold_uris | sibling_uris)].reset_index(drop=True)
print(f"Competenze corrette: {len(gold_uris)}, gruppi che le contengono: {gold_parents.nunique()}")
print(f"Campione: {len(taxonomy)} competenze ({len(taxonomy) / len(esco):.0%} di ESCO)")

Competenze corrette: 278, gruppi che le contengono: 216
Campione: 6291 competenze (45% di ESCO)


## 3. L'encoder: collegare un testo alla tassonomia

Trasformiamo in vettori sia il testo da collegare sia le etichette ESCO, e scegliamo l'etichetta **più vicina**
(similarità del coseno, cosine similarity): una **ricerca per significato** (semantic search).

Per valutare solo l'encoder usiamo le menzioni annotate a mano (la colonna `span`) e misuriamo:
- **Recall@1**: la quota di menzioni per cui l'etichetta corretta è la prima;
- **Recall@5**: la quota per cui è tra le prime cinque (nella sezione 5 il decoder sceglierà proprio tra le prime cinque);
- **MRR** (Mean Reciprocal Rank): la media di 1/posizione dell'etichetta corretta (1 se prima, 0,5 se seconda, ...).

In [9]:
pairs = mentions["validation"].query("label not in @NO_LABEL").drop_duplicates(["sentence", "span", "label"])
labels = taxonomy["preferredLabel"].to_numpy()
gold_index = pd.Series(range(len(labels)), index=labels)[pairs["label"]].to_numpy()


def retrieval_scores(similarity):
    """Recall@1, Recall@5 e MRR a partire dalla matrice delle similarità (menzioni x etichette)."""
    rank = (similarity > similarity[np.arange(len(similarity)), gold_index][:, None]).sum(1) + 1
    return {"Recall@1": (rank == 1).mean(), "Recall@5": (rank <= 5).mean(), "MRR": (1 / rank).mean()}


print(f"Menzioni di validazione con etichetta ESCO: {len(pairs)}")

Menzioni di validazione con etichetta ESCO: 265


### 3.1 Dalle rappresentazioni statiche agli encoder moderni

| Rappresentazione | Tipo | Idea |
|:--|:--|:--|
| TF-IDF | statica, lessicale | conta le parole in comune |
| GloVe (media dei vettori delle parole) | statica, semantica | parole simili hanno vettori vicini, ma ogni parola ha **un solo** vettore |
| `all-MiniLM-L6-v2` (2021, 22 milioni di parametri) | contestuale | un encoder di frasi piccolo e veloce |
| `granite-embedding-english-r2` (IBM, 2025, 149 milioni) | contestuale | un encoder moderno, addestrato soprattutto per la ricerca (retrieval) |
| `Qwen3-Embedding-0.6B` (Alibaba, 2025, 596 milioni) | contestuale | tra i migliori modelli sotto il miliardo di parametri nella classifica MTEB |

I due encoder moderni sono stati scelti consultando **MTEB** (Massive Text Embedding Benchmark), la classifica di riferimento
dei modelli di embedding: colonna **retrieval**, al massimo 600 milioni di parametri, licenza aperta. Qwen3-Embedding è un
LLM riadattato a produrre vettori: conta il **ruolo** (trasformare un testo in un vettore), non l'architettura.

In [10]:
mapping_results = {}

# 1. TF-IDF: vocabolario costruito su etichette e menzioni
tfidf = TfidfVectorizer().fit(list(labels) + pairs["span"].tolist())
mapping_results["TF-IDF"] = retrieval_scores((tfidf.transform(pairs["span"]) @ tfidf.transform(labels).T).toarray())

# 2. GloVe: media dei vettori delle parole (circa 130 MB da scaricare la prima volta)
glove = gensim.downloader.load("glove-wiki-gigaword-100")


def glove_embed(texts):
    vectors = []
    for text in texts:
        words = [w for w in re.findall(r"[a-z]+", text.lower()) if w in glove]
        vectors.append(np.mean([glove[w] for w in words], axis=0) if words else np.zeros(glove.vector_size))
    vectors = np.array(vectors)
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True).clip(1e-9)   # norma 1: il prodotto scalare è il coseno


mapping_results["GloVe"] = retrieval_scores(glove_embed(pairs["span"]) @ glove_embed(labels).T)


# 3. Encoder contestuali (normalize_embeddings=True: norma 1, come sopra)
def load_encoder(name):
    """Un encoder di frasi; sulla GPU in mezza precisione (più veloce, meno memoria)."""
    return SentenceTransformer(name, device=DEVICE, model_kwargs={"torch_dtype": torch.float16} if DEVICE == "cuda" else {})


def encoder_scores(model, prompt=None):
    """Recall@1, Recall@5 e MRR di un encoder; prompt: istruzione opzionale, aggiunta a menzioni ed etichette."""
    spans = model.encode(pairs["span"].tolist(), prompt=prompt, normalize_embeddings=True)
    return retrieval_scores(spans @ model.encode(list(labels), batch_size=128, prompt=prompt, normalize_embeddings=True).T)


minilm = load_encoder("sentence-transformers/all-MiniLM-L6-v2")
encoder = load_encoder("ibm-granite/granite-embedding-english-r2")
qwen_embedding = load_encoder("Qwen/Qwen3-Embedding-0.6B")
SKILL_INSTRUCTION = "Instruct: Given the name of a skill, retrieve skills with the same meaning\nQuery: "

mapping_results["MiniLM"] = encoder_scores(minilm)
mapping_results["granite-r2"] = encoder_scores(encoder)
mapping_results["Qwen3-Embedding"] = encoder_scores(qwen_embedding)
mapping_results["Qwen3-Embedding + istruzione"] = encoder_scores(qwen_embedding, SKILL_INSTRUCTION)
del qwen_embedding                                         # libera la memoria della GPU

pd.DataFrame(mapping_results).T.round(3)

,Recall@1,Recall@5,MRR
TF-IDF,0.483,0.717,0.586
GloVe,0.389,0.517,0.451
MiniLM,0.555,0.796,0.668
granite-r2,0.570,0.845,0.688
Qwen3-Embedding,0.551,0.758,0.649
Qwen3-Embedding + istruzione,0.611,0.834,0.719


- Gli **encoder contestuali** battono le rappresentazioni statiche; **GloVe** è il peggiore, sotto anche TF-IDF:
  la media dei vettori delle parole perde il senso dell'insieme, e ogni parola ha un solo vettore per tutti i suoi significati.
- **granite-r2** ha la Recall@5 migliore, cioè mette più spesso l'etichetta corretta tra i cinque candidati.
- **Qwen3-Embedding**, il più grande dei candidati e il più in alto in MTEB, senza istruzione non va oltre MiniLM;
  con un'**istruzione** che descrive il compito (la stessa su menzioni ed etichette, perché confrontiamo nomi con nomi)
  in prima posizione raggiunge i migliori.

Due lezioni: MTEB serve a scegliere i **candidati**, ma la decisione si prende sui **propri dati**;
e i modelli di embedding più recenti vanno usati con l'istruzione giusta.

Scegliamo **granite-r2**: il decoder sceglierà tra i primi cinque candidati, quindi conta la Recall@5;
ed è quattro volte più piccolo di Qwen3-Embedding.

#### Statico e contestuale da vicino: "python"

GloVe ha **un solo vettore** per "python", qualunque sia il contesto. MiniLM legge la frase intera.

In [11]:
glove_python = [w for w, _ in glove.most_similar("python", topn=8)]
print("GloVe, parole più vicine a 'python':", glove_python)

# MiniLM: similarità tra due frasi con "python" e due significati
contexts = ["we use python for data analysis", "the zoo keeper fed the python"]
meanings = ["Python (computer programming)", "snake"]
context_table = pd.DataFrame((minilm.encode(contexts, normalize_embeddings=True)
                              @ minilm.encode(meanings, normalize_embeddings=True).T).astype(float), index=contexts, columns=meanings)
context_table.round(2)

GloVe, parole più vicine a 'python': ['monty', 'php', 'perl', 'cleese', 'flipper', 'ruby', 'spamalot', 'javascript']


,Python (computer programming),snake
we use python for data analysis,0.69,0.23
the zoo keeper fed the python,0.27,0.45


- Per GloVe "python" è un unico vettore che mescola i Monty Python (monty, cleese, spamalot) e i linguaggi di programmazione.
- MiniLM legge la frase intera: la stessa parola è più vicina a "Python (computer programming)" quando si parla di dati,
  a "snake" quando si parla di zoo.

### 3.2 Cosa rappresentare della tassonomia?

Oltre all'etichetta preferita, ESCO offre **descrizioni** e **sinonimi**. Con granite-r2 proviamo:
- **etichetta + descrizione**: un solo testo, più lungo, per ogni competenza;
- **etichetta + sinonimi**: un vettore per ogni nome della competenza; conta la similarità **massima** tra i suoi nomi.

Il secondo modo è il nostro **indice**: una funzione lo costruisce per un insieme qualsiasi di competenze,
un'altra calcola la similarità tra dei testi e le competenze dell'indice.

In [12]:
def build_index(skills, model, synonyms=True):
    """Indice di una tassonomia: un vettore per ogni nome di ogni competenza (etichetta ed eventuali sinonimi)."""
    names_ = [[label] + (alts.split("\n") if synonyms and isinstance(alts, str) else [])
              for label, alts in zip(skills["preferredLabel"], skills["altLabels"])]
    starts = np.cumsum([0] + [len(n) for n in names_[:-1]])            # dove iniziano i nomi di ogni competenza
    vectors = model.encode([n for ns in names_ for n in ns], batch_size=256, normalize_embeddings=True)
    return {"model": model, "labels": skills["preferredLabel"].to_numpy(), "starts": starts, "vectors": vectors}


def similarity(texts, index):
    """Similarità tra testi e competenze: per ogni competenza vale il nome più simile."""
    S = index["model"].encode(list(texts), normalize_embeddings=True) @ index["vectors"].T
    return np.maximum.reduceat(S, index["starts"], axis=1)           # massimo su ogni blocco di nomi


start = time.time()
index = build_index(taxonomy, encoder)
print(f"Indice con sinonimi: {len(index['vectors'])} nomi per {len(labels)} competenze, calcolato in {time.time() - start:.0f} s")
mapping_results["granite-r2 + sinonimi"] = retrieval_scores(similarity(pairs["span"], index))

described = encoder.encode([f"{l}: {d}" for l, d in zip(labels, taxonomy["description"].fillna(""))],
                           batch_size=128, normalize_embeddings=True)
mapping_results["granite-r2 + descrizione"] = retrieval_scores(
    encoder.encode(pairs["span"].tolist(), normalize_embeddings=True) @ described.T)

mapping_table = pd.DataFrame(mapping_results).T.round(3)
fig = px.bar(mapping_table.reset_index(names="Rappresentazione").melt(id_vars="Rappresentazione"),
             x="Rappresentazione", y="value", color="variable", barmode="group", text_auto=".2f",
             title="Collegare le menzioni a ESCO (validazione)", labels={"value": "", "variable": "Metrica"})
fig.show()
mapping_table

Indice con sinonimi: 48715 nomi per 6291 competenze, calcolato in 6 s


,Recall@1,Recall@5,MRR
TF-IDF,0.483,0.717,0.586
GloVe,0.389,0.517,0.451
MiniLM,0.555,0.796,0.668
granite-r2,0.570,0.845,0.688
Qwen3-Embedding,0.551,0.758,0.649
Qwen3-Embedding + istruzione,0.611,0.834,0.719
granite-r2 + sinonimi,0.608,0.887,0.720
granite-r2 + descrizione,0.611,0.845,0.710


- **Sinonimi** e **descrizione** migliorano entrambi la prima posizione.
- Solo i **sinonimi** portano più spesso l'etichetta corretta tra i primi cinque candidati: è ciò che serve al decoder.

Da qui in poi l'encoder è **granite-r2 con l'indice dei sinonimi**.

#### Salvare l'indice

I vettori dei nomi sono l'**indice** della tassonomia: si calcolano una volta e si salvano insieme alle etichette
e alle posizioni dei blocchi. Salviamo anche l'encoder: si ricarica dalla cartella senza scaricarlo dall'Hub
(utile, per esempio, su un server aziendale senza internet).

In [13]:
np.savez_compressed(SAVE_DIR / "indice_esco_campione.npz",
                    vectors=index["vectors"], labels=index["labels"], starts=index["starts"])
encoder.save(str(SAVE_DIR / "encoder_granite"))
reloaded = SentenceTransformer(str(SAVE_DIR / "encoder_granite"), device=DEVICE)
same = np.allclose(reloaded.encode(["python"]), encoder.encode(["python"]), atol=1e-2)
print(f"Indice: {(SAVE_DIR / 'indice_esco_campione.npz').stat().st_size / 1e6:.1f} MB. Encoder ricaricato, stessi vettori: {same}")

Indice: 61.9 MB. Encoder ricaricato, stessi vettori: True


## 4. Il decoder: estrarre le competenze

### 4.1 Un client per tutti i modelli

**OpenRouter** dà accesso a centinaia di modelli con un'unica interfaccia, quella di OpenAI:
cambiare modello significa cambiare una stringa. Ogni richiesta ha tre accorgimenti:
- `temperature=0`: risposte il più possibile deterministiche (dove il modello lo permette);
- `response_format` con uno **schema JSON**: il modello deve rispondere in un formato che il codice sa leggere (structured output);
- `max_tokens`: un tetto alla lunghezza della risposta. **Indispensabile**: un modello può "incantarsi"
  e ripetere lo stesso testo fino al limite, facendo crescere tempo e costo.

Confrontiamo due decoder:
- **Qwen3 30B-A3B** (Alibaba, 2025): open-weight (i pesi sono pubblici: un'azienda può eseguirlo sui propri server)
  e **mixture of experts**: 30 miliardi di parametri, ma per ogni token ne lavorano circa 3. Molto economico.
- **Muse Ember** (Meta, novembre 2026): più recente e più forte, disponibile solo tramite API. È un modello di
  **ragionamento**: prima di rispondere genera dei token di "pensiero", che si pagano come output. Non si può disattivare:
  chiediamo lo sforzo minimo (`effort: "minimal"`) e un tetto di token più alto.

Lo abbiamo scelto con un pilota sulla validazione tra sei modelli economici recenti, preselezionati con Artificial Analysis.

In [14]:
DECODERS = {"Qwen3 30B-A3B": "qwen/qwen3-30b-a3b-instruct-2507",
            "Muse Ember": "meta/muse-ember"}
REASONING = {"meta/muse-ember": "minimal"}                       # sforzo di ragionamento, per i modelli che ragionano

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=API_KEY or "nessuna", timeout=90, max_retries=3)
cache_path = SAVE_DIR / ("risposte_llm.json" if USE_COURSE_CACHE else "risposte_llm_mie.json")   # le vostre in un file a parte
if USE_COURSE_CACHE and not cache_path.exists():                   # la cache del corso: le risposte già pronte
    urlretrieve(f"{COURSE}/materiale/risposte_llm.json", cache_path)
cache = json.loads(cache_path.read_text()) if cache_path.exists() else {}
usage = {"chiamate": 0, "costo_dollari": 0.0, "troncate": 0, "mancanti": 0}
lock = threading.Lock()


def ask(model, instructions, text, schema, max_tokens=800):
    """Una richiesta al modello, con risposta JSON vincolata allo schema. Le risposte già ottenute vengono dalla cache."""
    key = json.dumps([model, instructions, text, schema])
    if key not in cache:
        if not API_KEY:                                                   # senza chiave la risposta resta mancante
            with lock:
                usage["mancanti"] += 1
            return None
        extra = {"usage": {"include": True},                              # OpenRouter restituisce il costo della chiamata
                 "provider": {"require_parameters": True}}                # solo fornitori che rispettano lo schema
        if model in REASONING:
            extra["reasoning"] = {"effort": REASONING[model]}
        response = client.chat.completions.create(
            model=model, temperature=0, max_tokens=max_tokens,
            messages=[{"role": "system", "content": instructions}, {"role": "user", "content": text}],
            response_format={"type": "json_schema", "json_schema": {"name": "risposta", "strict": True, "schema": schema}},
            extra_body=extra)
        with lock:
            usage["chiamate"] += 1
            usage["costo_dollari"] += getattr(response.usage, "cost", 0) or 0
            usage["troncate"] += response.choices[0].finish_reason == "length"
        cache[key] = response.choices[0].message.content
    try:
        return json.loads(cache[key])
    except (json.JSONDecodeError, TypeError):                             # risposta troncata o vuota: la consideriamo assente
        return None


def ask_many(model, instructions, texts, schema, max_tokens=800):
    """Molte richieste in parallelo (8 alla volta); alla fine la cache viene salvata su file."""
    with ThreadPoolExecutor(8) as pool:
        answers = list(pool.map(lambda t: ask(model, instructions, t, schema, max_tokens), texts))
    cache_path.write_text(json.dumps(cache))
    return answers

### 4.2 Le istruzioni e lo schema

Per ogni competenza chiediamo due campi:
- `span`: le parole **copiate** dalla frase (così possiamo verificare che il modello non inventi);
- `skill`: un **nome breve e generico** in inglese, come quelli di una tassonomia.

Nella sezione 5 vedremo quale dei due conviene dare all'encoder.

In [15]:
EXTRACT_INSTRUCTIONS = (
    "You extract skills from job advertisement sentences. A skill is any ability, competence, knowledge area, "
    "tool or technology the candidate is expected to have or use. For each skill return 'span': the exact words "
    "copied from the sentence, and 'skill': a short, generic name for it in English (e.g. 'manage a team'). "
    "Ignore degrees, years of experience and generic personality traits without a concrete skill. "
    "If there are no skills, return an empty list.")

EXTRACT_SCHEMA = {
    "type": "object", "additionalProperties": False, "required": ["skills"],
    "properties": {"skills": {"type": "array", "items": {
        "type": "object", "additionalProperties": False, "required": ["span", "skill"],
        "properties": {"span": {"type": "string"}, "skill": {"type": "string"}}}}}}

example = "Strong knowledge of SQL and experience leading a small team of analysts."
example_extractions = {name: ask(model, EXTRACT_INSTRUCTIONS, example, EXTRACT_SCHEMA) for name, model in DECODERS.items()}
for name, answer in example_extractions.items():
    print(name, json.dumps(answer, indent=1))

Qwen3 30B-A3B {
 "skills": [
  {
   "span": "Strong knowledge of SQL",
   "skill": "SQL"
  },
  {
   "span": "experience leading a small team of analysts",
   "skill": "lead a team"
  }
 ]
}
Muse Ember {
 "skills": [
  {
   "skill": "SQL",
   "span": "SQL"
  },
  {
   "skill": "lead a team",
   "span": "leading a small team of analysts"
  }
 ]
}


### 4.3 Quanto estraggono bene?

Sulle frasi di validazione confrontiamo le competenze estratte con le menzioni annotate a mano:
- **copertura**: la quota di menzioni annotate che il modello ritrova (basta che un testo contenga l'altro);
- **aderenza al testo** (grounding): la quota di `span` estratti che compaiono davvero nella frase.

In [16]:
def normalize(s):
    return " ".join(re.sub(r"[^\w+#]", " ", s.lower()).split())


def skills_of(answer):
    """Le competenze di una risposta di estrazione (lista vuota se la risposta manca)."""
    return (answer or {"skills": []})["skills"]


def extraction_quality(answers, sentences):
    found = total = grounded = extracted = 0
    for answer, (sentence, spans) in zip(answers, sentences.items()):
        predicted = [normalize(s["span"]) for s in skills_of(answer)]
        extracted += len(predicted)
        grounded += sum(p in normalize(sentence) for p in predicted)
        for span in set(spans):
            total += 1
            found += any(p and (p in normalize(span) or normalize(span) in p) for p in predicted)
    return {"Copertura delle menzioni": found / total, "Aderenza al testo": grounded / max(extracted, 1),
            "Competenze estratte": extracted}


val_spans = mentions["validation"].groupby("sentence", sort=False)["span"].agg(list)[val["sentence"]]
extractions, quality = {}, {}
for name, model in DECODERS.items():
    start, cost_before = time.time(), usage["costo_dollari"]
    extractions[name] = {split: ask_many(model, EXTRACT_INSTRUCTIONS, data["sentence"].tolist(), EXTRACT_SCHEMA)
                         for split, data in SETS.items()}
    quality[name] = {**extraction_quality(extractions[name]["val"], val_spans),
                     "Tempo (s)": time.time() - start, "Costo ($)": usage["costo_dollari"] - cost_before}

pd.DataFrame(quality).T.round(3)

,Copertura delle menzioni,Aderenza al testo,Competenze estratte,Tempo (s),Costo ($)
Qwen3 30B-A3B,0.871,0.982,393.0,79.875,0.010
Muse Ember,0.836,0.998,404.0,41.751,0.029


- **Qwen3 30B-A3B** ritrova qualche menzione in più; **Muse Ember** copia lo span quasi sempre alla lettera.
- Muse Ember costa circa tre volte di più, ma restiamo nell'ordine dei centesimi.

Quale decoder conviene davvero lo dice l'intera pipeline (sezione 5).

## 5. Quattro strategie a confronto

Valutiamo l'intero processo, dalla frase alle etichette ESCO, sulle **150 frasi di test**.
Per ogni frase confrontiamo l'insieme di etichette previste con quello corretto:
- **precisione** (precision): la quota di etichette previste che sono corrette;
- **richiamo** (recall): la quota di etichette corrette che sono state trovate;
- **F1**: la loro media armonica.

| Strategia | Decoder | Encoder |
|:--|:--|:--|
| A. Solo encoder | no | confronta la frase intera con la tassonomia |
| B. Solo decoder | scrive direttamente le etichette ESCO | no |
| C. Estrai e collega | estrae le competenze | collega ciascuna alla voce più vicina |
| D. Estrai, collega e scegli | estrae, poi sceglie tra 5 candidati | propone i 5 candidati più vicini |

Decoder: **Muse Ember**; encoder: **granite-r2 con sinonimi**. Alla fine misuriamo quanto valgono questi due modelli
rispetto a quelli più piccoli.

In [17]:
MAIN = "Muse Ember"                                        # il decoder principale


def prf(gold_sets, predicted_sets):
    """Precisione, richiamo e F1 sommando su tutte le frasi (micro-media)."""
    tp = sum(len(g & p) for g, p in zip(gold_sets, predicted_sets))
    n_pred, n_gold = sum(map(len, predicted_sets)), sum(map(len, gold_sets))
    precision, recall = tp / max(n_pred, 1), tp / max(n_gold, 1)
    return {"Precisione": precision, "Richiamo": recall,
            "F1": 2 * precision * recall / max(precision + recall, 1e-9), "Etichette previste": n_pred}


def best_threshold(predict):
    """La soglia di similarità con la F1 migliore sulla validazione. predict(split, soglia) -> etichette previste."""
    scores = {t: prf(val["gold"], predict("val", t))["F1"] for t in np.arange(0.3, 0.95, 0.05).round(2)}
    return max(scores, key=scores.get)


strategies = {}

### A. Solo encoder

Confrontiamo la frase intera con la tassonomia e teniamo fino a 3 etichette, solo se abbastanza simili:
la **soglia** di similarità si sceglie sulla validazione.

In [18]:
sentence_similarity = {split: similarity(data["sentence"], index) for split, data in SETS.items()}


def strategy_a(split, threshold):
    S = sentence_similarity[split]
    top3 = np.argsort(-S, axis=1)[:, :3]
    return [{index["labels"][j] for j in row if S[i, j] >= threshold} for i, row in enumerate(top3)]


t_a = best_threshold(strategy_a)
strategies["A. Solo encoder"] = prf(test["gold"], strategy_a("test", t_a))
print(f"Soglia scelta: {t_a}")

Soglia scelta: 0.85


### B. Solo decoder

Chiediamo al decoder di scrivere direttamente le etichette ESCO ufficiali, senza dargli la tassonomia:
13.896 voci in ogni richiesta costerebbero molto. Deve ricordarle.

Siamo **generosi**: se un'etichetta scritta coincide con un nome ESCO a meno di maiuscole e minuscole,
o con uno dei suoi sinonimi, la convertiamo nell'etichetta ufficiale prima di valutarla.

In [19]:
DIRECT_INSTRUCTIONS = ("You map job advertisement sentences to the ESCO skills taxonomy (v1.1). Return the ESCO skills "
                       "required by the sentence, using their exact official ESCO preferred labels. "
                       "If none applies, return an empty list.")
DIRECT_SCHEMA = {"type": "object", "additionalProperties": False, "required": ["esco_skills"],
                 "properties": {"esco_skills": {"type": "array", "items": {"type": "string"}}}}

direct = ask_many(DECODERS[MAIN], DIRECT_INSTRUCTIONS, test["sentence"].tolist(), DIRECT_SCHEMA)
written = [(a or {"esco_skills": []})["esco_skills"] for a in direct]

# Ogni nome ESCO (etichetta o sinonimo, in minuscolo) punta all'etichetta ufficiale
official = {alt.strip().lower(): label for label, alts in zip(esco["preferredLabel"], esco["altLabels"])
            if isinstance(alts, str) for alt in alts.split("\n")}
official.update({label.lower(): label for label in esco["preferredLabel"]})

predicted_b = [{official.get(l.strip().lower(), l) for l in labels_b} for labels_b in written]
strategies["B. Solo decoder"] = prf(test["gold"], predicted_b)

all_written = [l for labels_b in written for l in labels_b]
exact = np.mean([l in set(esco["preferredLabel"]) for l in all_written])
lenient = np.mean([l.strip().lower() in official for l in all_written])
invented = [l for l in all_written if l.strip().lower() not in official]
print(f"Etichette scritte: {len(all_written)}")
print(f"  identiche a un'etichetta ESCO: {exact:.0%}")
print(f"  riconoscibili in ESCO ignorando le maiuscole o come sinonimo: {lenient:.0%}")
print("Esempi di etichette che in ESCO non esistono in nessuna forma:", invented[:8])

Etichette scritte: 298
  identiche a un'etichetta ESCO: 14%
  riconoscibili in ESCO ignorando le maiuscole o come sinonimo: 27%
Esempi di etichette che in ESCO non esistono in nessuna forma: ['write academic papers', 'lead product development', 'leadership', 'sales and operations planning', 'communicate effectively', 'data visualisation', 'public speaking', 'use product lifecycle management software']


### C. Estrai e collega

Il decoder estrae le competenze (sezione 4), l'encoder collega ciascuna alla voce più vicina.
Se la similarità è sotto la soglia la risposta è "nessuna voce": è il caso `LABEL NOT PRESENT`.

Quale testo dare all'encoder: le parole copiate dalla frase (`span`) o il nome generico scritto dal decoder (`skill`)?
Lo decidiamo sulla **validazione**, insieme alla soglia, per entrambi i decoder.

In [20]:
def mentions_of(answer, field):
    """I testi estratti da una risposta del decoder (campo 'span' o 'skill')."""
    return [s[field] for s in skills_of(answer) if s[field].strip()]


def best_matches(answers, field, index):
    """Per ogni frase: la voce più vicina a ciascuna menzione estratta, con la similarità."""
    matches = []
    for answer in answers:
        texts = mentions_of(answer, field)
        S = similarity(texts, index) if texts else np.zeros((0, len(index["labels"])))
        matches.append(list(zip(index["labels"][S.argmax(1)], S.max(1))))
    return matches


matches = {(d, f, split): best_matches(extractions[d][split], f, index)
           for d in DECODERS for f in ["span", "skill"] for split in SETS}


def strategy_c(split, threshold, field, decoder=MAIN):
    return [{label for label, s in m if s >= threshold} for m in matches[(decoder, field, split)]]


choices_c = []
for decoder in DECODERS:
    for field in ["span", "skill"]:
        t = best_threshold(lambda split, th: strategy_c(split, th, field, decoder))
        choices_c.append({"Decoder": decoder, "Testo": field, "Soglia": t,
                          "F1 validazione": prf(val["gold"], strategy_c("val", t, field, decoder))["F1"]})
choices_c = pd.DataFrame(choices_c)
choices_c.round(3)

,Decoder,Testo,Soglia,F1 validazione
0,Qwen3 30B-A3B,span,0.85,0.388
1,Qwen3 30B-A3B,skill,0.90,0.369
2,Muse Ember,span,0.90,0.402
3,Muse Ember,skill,0.90,0.416


Il testo migliore dipende dal decoder, e le differenze sono piccole: lo scegliamo sulla validazione **per ciascun decoder**.
Il campo `skill` serve comunque per l'italiano (sezione 8).

In [21]:
best_c = {d: choices_c.query("Decoder == @d").sort_values("F1 validazione").iloc[-1] for d in DECODERS}
FIELDS = {d: row["Testo"] for d, row in best_c.items()}            # il testo da collegare, per ciascun decoder
t_c = best_c[MAIN]["Soglia"]
strategies["C. Estrai e collega"] = prf(test["gold"], strategy_c("test", t_c, FIELDS[MAIN]))
print("Testo scelto per ciascun decoder:", FIELDS, f"  soglia di {MAIN}: {t_c}")

Testo scelto per ciascun decoder: {'Qwen3 30B-A3B': 'span', 'Muse Ember': 'skill'}   soglia di Muse Ember: 0.9


### D. Estrai, collega e scegli

L'encoder propone i **5 candidati** più vicini e il decoder, leggendo la frase, sceglie quello giusto o nessuno.
L'encoder **cerca** in fretta tra migliaia di voci, il decoder **ragiona** su poche: è l'idea dei sistemi RAG
(retrieval-augmented generation), dove il modello generativo decide guardando solo ciò che è stato recuperato.

In [22]:
CHOOSE_INSTRUCTIONS = ("You link a skill mentioned in a job advertisement to the ESCO taxonomy. You get the sentence, "
                       "the skill and a numbered list of candidate ESCO skills. Return the number of the candidate that "
                       "matches the skill, or 0 if none of them matches.")
CHOOSE_SCHEMA = {"type": "object", "additionalProperties": False, "required": ["choice"],
                 "properties": {"choice": {"type": "integer"}}}


def choose_request(sentence, mention, candidates):
    return (f"Sentence: {sentence}\nSkill: {mention}\nCandidates:\n" +
            "\n".join(f"{n}. {c}" for n, c in enumerate(candidates, 1)))


def strategy_d(split, index, decoder=MAIN, k=5):
    requests, owners = [], []
    for i, (answer, sentence) in enumerate(zip(extractions[decoder][split], SETS[split]["sentence"])):
        found = mentions_of(answer, FIELDS[decoder])
        if not found:
            continue
        top = np.argsort(-similarity(found, index), axis=1)[:, :k]
        for mention, row in zip(found, top):
            requests.append(choose_request(sentence, mention, index["labels"][row]))
            owners.append((i, index["labels"][row]))
    choices = ask_many(DECODERS[decoder], CHOOSE_INSTRUCTIONS, requests, CHOOSE_SCHEMA, max_tokens=400)
    predicted = [set() for _ in range(len(SETS[split]))]
    for (i, candidates), choice in zip(owners, choices):
        if choice and 1 <= choice["choice"] <= k:
            predicted[i].add(candidates[choice["choice"] - 1])
    return predicted


predicted_d = strategy_d("test", index)
strategies["D. Estrai, collega e scegli"] = prf(test["gold"], predicted_d)

#### Una frase attraverso tutta la pipeline
La frase di esempio della sezione 4: le menzioni estratte, i 5 candidati dell'encoder (con la similarità)
e la scelta del decoder, che non è sempre il primo candidato.

In [23]:
walkthrough = []
for mention in mentions_of(example_extractions[MAIN], FIELDS[MAIN]):
    sims = similarity([mention], index)[0]
    top = np.argsort(-sims)[:5]
    answer = ask(DECODERS[MAIN], CHOOSE_INSTRUCTIONS, choose_request(example, mention, index["labels"][top]),
                 CHOOSE_SCHEMA, 400)
    choice = (answer or {"choice": 0})["choice"]
    walkthrough.append({"Menzione": mention,
                        "Candidati (similarità)": [f"{index['labels'][j]} ({sims[j]:.2f})" for j in top],
                        "Scelta del decoder": index["labels"][top[choice - 1]] if 1 <= choice <= 5 else "nessuna"})
with pd.option_context("display.max_colwidth", 300):
    display(pd.DataFrame(walkthrough))

,Menzione,Candidati (similarità),Scelta del decoder
0,SQL,"[database management systems (1.00), SQL (1.00), SQL Server (0.94), MySQL (0.89), query languages (0.89)]",SQL
1,lead a team,"[lead a team (1.00), manage a team (0.92), lead a team in hospitality service (0.91), direct an artistic team (0.91), manage a social work unit (0.90)]",lead a team


In [24]:
comparison = pd.DataFrame(strategies).T
fig = px.bar(comparison.reset_index(names="Strategia").melt(id_vars="Strategia", value_vars=["Precisione", "Richiamo", "F1"]),
             x="Strategia", y="value", color="variable", barmode="group", text_auto=".2f",
             title=f"Dalla frase alle etichette ESCO: test, {len(test)} frasi (decoder: {MAIN})",
             labels={"value": "", "variable": "Metrica"})
fig.show()
comparison.round(3)

,Precisione,Richiamo,F1,Etichette previste
A. Solo encoder,0.225,0.260,0.241,249.0
B. Solo decoder,0.135,0.186,0.156,297.0
C. Estrai e collega,0.464,0.386,0.421,179.0
"D. Estrai, collega e scegli",0.405,0.498,0.447,264.0


- **A. Solo encoder**: la frase intera assomiglia a molte competenze insieme; la precisione è bassa.
- **B. Solo decoder**: anche con una valutazione generosa, la grande maggioranza delle etichette scritte non esiste in ESCO.
  Sembrano giuste, ma non hanno una voce: è un'**allucinazione**, e non si possono contare né confrontare.
- **C. Estrai e collega**: il decoder isola le menzioni, l'encoder le ancora alla tassonomia. Il salto è netto.
- **D. Estrai, collega e scegli**: il decoder sceglie leggendo la frase e recupera competenze che la soglia di C scartava:
  il richiamo sale e la F1 è la migliore.

**Perché i numeri assoluti non sono più alti?** Ogni menzione ha **una** etichetta corretta tra migliaia, e spesso più voci ESCO
sono ragionevoli: una scelta diversa da quella dell'annotatore conta come errore.

#### Quanto contano i modelli?

La stessa strategia D con tre combinazioni: i modelli di partenza (Qwen3 30B-A3B e MiniLM), solo il decoder più forte,
e decoder ed encoder più forti. Per MiniLM usiamo un indice con le sole etichette, come all'inizio della sezione 3;
ogni decoder usa il testo scelto per lui sulla validazione.

In [25]:
minilm_index = build_index(taxonomy, minilm, synonyms=False)
setups = {"Qwen3 30B-A3B + MiniLM": ("Qwen3 30B-A3B", minilm_index),
          "Muse Ember + MiniLM": ("Muse Ember", minilm_index),
          "Muse Ember + granite-r2 con sinonimi": ("Muse Ember", index)}
model_effect = {name: prf(test["gold"], strategy_d("test", idx, decoder)) for name, (decoder, idx) in setups.items()}
pd.DataFrame(model_effect).T.round(3)

,Precisione,Richiamo,F1,Etichette previste
Qwen3 30B-A3B + MiniLM,0.370,0.442,0.403,257.0
Muse Ember + MiniLM,0.418,0.488,0.451,251.0
Muse Ember + granite-r2 con sinonimi,0.405,0.498,0.447,264.0


- Il guadagno viene dal **decoder** più forte.
- L'**encoder** più forte, nella strategia D, non cambia il risultato: la differenza è dentro il rumore.
- Attenzione al **rumore**: con un modello di ragionamento le risposte cambiano un po' da un'esecuzione all'altra,
  anche con `temperature=0`. Differenze di 0,01-0,03 di F1 non sono affidabili.

Dove invece l'encoder decide da solo, come nella strategia C, l'encoder più forte aiuta:

In [26]:
c_effect = {}
for name, idx in {"MiniLM": minilm_index, "granite-r2 con sinonimi": index}.items():
    found = {split: best_matches(extractions[MAIN][split], FIELDS[MAIN], idx) for split in SETS}
    predict = lambda split, th: [{label for label, sim in m if sim >= th} for m in found[split]]
    c_effect[f"C con {name}"] = prf(test["gold"], predict("test", best_threshold(predict)))
pd.DataFrame(c_effect).T.round(3)

,Precisione,Richiamo,F1,Etichette previste
C con MiniLM,0.452,0.326,0.378,155.0
C con granite-r2 con sinonimi,0.464,0.386,0.421,179.0


Un componente migliore non sempre migliora il sistema: si misura sempre **l'intera pipeline**.

#### Da vicino
Tre frasi di test: confrontate le etichette corrette con quelle di C e D. Spesso l'etichetta "sbagliata" è una voce
ragionevole, diversa da quella scelta dall'annotatore.

In [27]:
view = test.assign(C=strategy_c("test", t_c, FIELDS[MAIN]), D=predicted_d)
view = view[view["gold"].str.len() > 0].sample(3, random_state=3)[["sentence", "gold", "C", "D"]]
with pd.option_context("display.max_colwidth", 200):
    display(view)

,sentence,gold,C,D
107,Our games are developed using Unity so we expect all our programmers to have solid knowledge of mobile game development in Unity3D and C# .,"{C#, Unity (digital game creation systems)}","{C#, Unity (digital game creation systems)}","{C#, Unity (digital game creation systems)}"
9,You thrive in an outgoing role with a broad interface right across the organisation and are able to communicate concisely and accurately with colleagues at all levels .,{communication},{communication},{communication}
10,Furthermore excellence in at least one of the following categories is highly desirable: data visualisation tools cloud platforms machine learning techniques and algorithms .,"{cloud technologies, algorithms, prepare visual data, machine learning}","{cloud technologies, algorithms, machine learning, deliver visual presentation of data}","{cloud technologies, algorithms, machine learning, deliver visual presentation of data}"


## 6. Dalle competenze alle feature

Ogni frase è diventata un insieme di **voci ESCO**: feature esplicite, interpretabili e confrontabili.
Risalendo la gerarchia le aggreghiamo al livello che serve: qui contiamo i **gruppi** più citati in ciascun dataset.
Attenzione: sono previsioni della strategia D, con i suoi errori.

In [28]:
features = (test.assign(skill=predicted_d).explode("skill").dropna(subset=["skill"])
            .assign(group=lambda d: d["skill"].map(lambda l: names[parents[label_to_uri[l]]])))

top_groups = (features.groupby(["source", "group"]).size().rename("Menzioni").reset_index()
              .sort_values("Menzioni", ascending=False).groupby("source").head(6))
fig = px.bar(top_groups, x="Menzioni", y="group", color="source", facet_col="source", orientation="h",
             title="I gruppi ESCO più citati nelle frasi di test (previsioni)", labels={"group": ""})
fig.update_yaxes(matches=None, showticklabels=True)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(showlegend=False)
fig.show()

La stessa informazione diventa una **matrice di feature** (una riga per frase, una colonna per gruppo ESCO),
pronta per un modello di machine learning: clustering di annunci, matching con i CV, analisi del mercato del lavoro.

In [29]:
feature_matrix = pd.crosstab(features["sentence"], features["group"]).clip(upper=1)
print(f"Matrice: {feature_matrix.shape[0]} frasi x {feature_matrix.shape[1]} gruppi ESCO")
feature_matrix.iloc[:5, :8]

Matrice: 123 frasi x 93 gruppi ESCO


group,adapt to change,advising on business or operational matters,advising on design or use of technologies,"advising on legal, regulatory or procedural matters",advocating for individual or community needs,allocating and controlling physical resources,analysing and evaluating information and data,analysing business operations
sentence,,,,,,,,
* Expertise in application data and infrastructure architecture & design disciplines,0,0,0,0,0,0,0,0
* Knowledge of Big Data/NoSql solutions is beneficial,0,0,0,0,0,0,0,0
* Knowledge of industry wide technology strategies and best practices,0,0,0,0,0,0,0,0
Ability to establish rapport and bond with a business partner The ability to think outside the box The ability to clearly and professionally present information <ORGANIZATION> is an enthusiastic team of accomplished tech professionals based in <LOCATION> <LOCATION> .,0,0,0,0,0,0,0,0
About you You have most likely worked in large enterprises being part of large scale system implementaions a a lead consultant You have had an exposure to Teamcenter or any other PLM enterprise system and understands how systems interact with ERP and manufacturing execution systems .,0,0,1,0,0,0,0,0


## 7. Tutta ESCO

Ripetiamo la strategia D con **tutte le 13.896 competenze**: basta costruire l'indice completo, con tutti i sinonimi
(circa un minuto con la GPU). Le estrazioni sono già fatte; servono solo le nuove scelte del decoder.

In [30]:
start = time.time()
full_index = build_index(esco, encoder)
print(f"Indice di tutta ESCO: {len(full_index['vectors'])} nomi, calcolato in {time.time() - start:.0f} s")
np.savez_compressed(SAVE_DIR / "indice_esco_completo.npz",
                    vectors=full_index["vectors"], labels=full_index["labels"], starts=full_index["starts"])

strategies["D. Estrai, collega e scegli (tutta ESCO)"] = prf(test["gold"], strategy_d("test", full_index))
pd.DataFrame(strategies).T.round(3).iloc[-2:]

Indice di tutta ESCO: 111379 nomi, calcolato in 14 s


,Precisione,Richiamo,F1,Etichette previste
"D. Estrai, collega e scegli",0.405,0.498,0.447,264.0
"D. Estrai, collega e scegli (tutta ESCO)",0.389,0.479,0.429,265.0


Con tutta ESCO la F1 cala di poco: le **sorelle** erano le concorrenti più difficili, ma non le uniche.
Il campione dà un'idea corretta del compito, un po' ottimista: i risultati si confermano sempre sulla tassonomia intera.

## 8. Bonus: un annuncio in italiano

ESCO esiste in 28 lingue, ma qui usiamo l'indice inglese (ora quello completo) e un encoder addestrato sull'inglese.
Lo `span` è in italiano: colleghiamo invece il campo `skill`, che il decoder scrive **in inglese**.
Il decoder fa anche da **traduttore**, l'encoder resta lo stesso.

In [31]:
italian_ad = ["Cerchiamo un addetto alla contabilità con buona conoscenza di Excel e della fatturazione elettronica.",
              "Il candidato gestirà il magazzino e coordinerà un piccolo gruppo di carrellisti.",
              "Richiesta esperienza nello sviluppo di applicazioni web con React e Node.js."]
answers_it = ask_many(DECODERS[MAIN], EXTRACT_INSTRUCTIONS, italian_ad, EXTRACT_SCHEMA)

rows = []
for sentence, answer in zip(italian_ad, answers_it):
    for s in skills_of(answer):
        sims = similarity([s["skill"]], full_index)[0]
        rows.append({"Frase": sentence[:45] + "...", "Span": s["span"], "Skill (EN)": s["skill"],
                     "Candidati ESCO (encoder)": list(full_index["labels"][np.argsort(-sims)[:3]])})
italian_table = pd.DataFrame(rows)
italian_table

,Frase,Span,Skill (EN),Candidati ESCO (encoder)
0,Cerchiamo un addetto alla contabilità con buo...,contabilità,accounting,"[accounting, accounting entries, accounting department processes]"
1,Cerchiamo un addetto alla contabilità con buo...,Excel,Excel,"[use spreadsheets software, use mathematical tools and equipment, office software]"
2,Cerchiamo un addetto alla contabilità con buo...,fatturazione elettronica,electronic invoicing,"[monitor billing procedures, issue sales invoices, record healthcare users' billing information]"
3,Il candidato gestirà il magazzino e coordiner...,gestirà il magazzino,warehouse management,"[perform warehousing operations, warehouse operations, manage warehouse operations]"
4,Il candidato gestirà il magazzino e coordiner...,coordinerà un piccolo gruppo di carrellisti,coordinate a team of forklift drivers,"[operate forklift, operation of lifting gears, drive agricultural machines]"
5,Richiesta esperienza nello sviluppo di applic...,sviluppo di applicazioni web,web application development,"[web programming, systems development life-cycle, application process]"
6,Richiesta esperienza nello sviluppo di applic...,React,React,"[relate empathetically, react to events in time-critical environments, think quickly]"
7,Richiesta esperienza nello sviluppo di applic...,Node.js,Node.js,"[JavaScript, JavaScript Framework, TypeScript]"


- Il decoder traduce le menzioni in nomi inglesi e l'encoder trova voci ESCO in gran parte pertinenti.
- Si vedono anche i **limiti della tassonomia**: React e Node.js **non hanno una voce** in ESCO v1.1.1.
  L'encoder propone comunque il candidato più vicino: è il passo di scelta (o la soglia) a dover rispondere "nessuna".

## 9. Costi e salvataggi

In [32]:
print(f"Chiamate al decoder in questa sessione: {usage['chiamate']} (le altre venivano dalla cache)")
print(f"Costo di questa sessione: {usage['costo_dollari']:.4f} dollari. Risposte troncate dal limite di token: {usage['troncate']}")
print(f"Risposte in cache: {len(cache)}")
if usage["mancanti"]:
    print(f"Richieste non in cache, saltate perché manca la chiave: {usage['mancanti']} (i risultati possono differire di poco)")
pd.Series({f.name: f"{f.stat().st_size / 1e6:.1f} MB" for f in sorted(SAVE_DIR.iterdir()) if f.is_file()}, name="File salvati")

Chiamate al decoder in questa sessione: 2109 (le altre venivano dalla cache)
Costo di questa sessione: 0.1332 dollari. Risposte troncate dal limite di token: 0
Risposte in cache: 2105


esco_v1.1.1_en.zip            7.9 MB
indice_esco_campione.npz     61.9 MB
indice_esco_completo.npz    141.9 MB
risposte_llm.json             1.9 MB
Name: File salvati, dtype: str

In [33]:
summary = {"mapping": mapping_table.to_dict(), "decoder_qualita": quality, "strategie": strategies,
           "modelli": model_effect, "encoder_in_c": c_effect, "soglie": {"A": t_a, "C": t_c},
           "campione": {"competenze": len(taxonomy), "esco": len(esco), "nomi_campione": len(index["vectors"]),
                        "nomi_esco": len(full_index["vectors"])},
           "uso": usage, "scelte_c": choices_c.to_dict("records"),
           "esempio": {"frase": example, "estrazioni": example_extractions, "percorso": walkthrough},
           "python": {"glove": glove_python, "contesti": context_table.round(2).to_dict("index")},
           "inventate": invented[:15], "etichette_scritte": len(all_written), "esatte": float(exact),
           "riconoscibili": float(lenient), "gruppi": top_groups.to_dict("records"),
           "italiano": italian_table.to_dict("records"), "matrice": list(feature_matrix.shape),
           "frasi": {"val": len(val), "test": len(test)}, "menzioni_val": len(pairs),
           "da_vicino": view.map(lambda v: sorted(v) if isinstance(v, set) else v).to_dict("records")}
(SAVE_DIR / "risultati_esercizio_2.json").write_text(json.dumps(summary, indent=1, default=float))
print("Salvato:", SAVE_DIR / "risultati_esercizio_2.json")

Salvato: salvataggi/risultati_esercizio_2.json


## 10. Cosa portare a casa

1. Una **tassonomia** trasforma il testo libero in **feature esplicite**: voci che si contano, si aggregano e si confrontano.
2. **Encoder e decoder** hanno mestieri diversi: l'encoder cerca in fretta tra migliaia di voci, il decoder legge,
   estrae e sceglie tra poche opzioni.
3. Il decoder **da solo inventa** etichette plausibili: va ancorato alla tassonomia.
4. Un **decoder più forte** migliora la pipeline, un encoder più forte solo dove decide da solo (strategia C). Le classifiche (MTEB, Artificial Analysis) indicano i candidati,
   il pilota sui propri dati decide.
5. Schema JSON, `max_tokens` e **cache** rendono un LLM un componente controllabile ed economico di una pipeline.
6. Guardare **gli errori**, non solo il numero: spesso rivelano una valutazione severa o una tassonomia incompleta.

**Per il project work**: sostituite le frasi con i vostri testi (ticket, CV, schede prodotto) e ESCO con la tassonomia
del vostro dominio. Cambiano i dati e le istruzioni del decoder: la struttura del codice resta la stessa.